# Horse Racing Ranking Model
**LambdaRank + Optuna + Calibrated Probabilities + Value Betting**

## How to use:
1. Upload your CSV (e.g. `all_tracks_hackathon_2016_2026.csv`) to `/content/` or Google Drive
2. Run cells in order
3. Model artifacts save to Google Drive automatically

## What it does:
- Trains a LightGBM LambdaRank model that learns horse ordering within races
- Optuna finds best hyperparameters (NDCG@3 optimization)
- Calibrates probabilities with isotonic regression
- Runs a linear regression baseline for comparison
- Evaluates with ROI simulation on historical odds
- Generates diagnostic plots + SHAP analysis

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install lightgbm optuna scikit-learn scipy joblib shap -q

import sys
sys.stdout.reconfigure(line_buffering=True)  # Force live output in Colab

In [ ]:
# ── Cell 2: Mount Google Drive & set paths ──
from google.colab import drive
drive.mount('/content/drive')

# ── CONFIGURE THIS ──
CSV_PATH = "/content/all_tracks_hackathon_2016_2026.csv"  # or your Drive path
OUTPUT_DIR = "/content/drive/MyDrive/horse_model"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input:  {CSV_PATH}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# ── Cell 3: Imports & helpers ──
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import re
import json
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from itertools import permutations, combinations
from scipy.special import softmax
from sklearn.model_selection import GroupKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    log_loss, brier_score_loss, mean_absolute_error, ndcg_score, mean_squared_error
)
from sklearn.calibration import calibration_curve
import shap

warnings.filterwarnings('ignore', category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.INFO)

_JT_LOOKUPS = {'jockey_trainer': {}, 'jockey': {}, 'trainer': {}}

def parse_time_to_seconds(time_str):
    if pd.isna(time_str): return np.nan
    if isinstance(time_str, (int, float)): return float(time_str)
    time_str = str(time_str).strip()
    m = re.match(r'(\d{1,2}):(\d{1,2}(?:\.\d+)?)', time_str)
    if m: return float(m.group(1)) * 60 + float(m.group(2))
    m = re.match(r'(\d+(?:\.\d+)?)', time_str)
    if m: return float(m.group(1))
    return np.nan

def calculate_days_between(d1, d2):
    if pd.isna(d1) or pd.isna(d2): return np.nan
    d1, d2 = pd.to_datetime(d1, errors='coerce'), pd.to_datetime(d2, errors='coerce')
    if pd.isna(d1) or pd.isna(d2): return np.nan
    return (d1 - d2).days

def scores_to_probs(scores, group_sizes):
    probs = np.zeros_like(scores)
    idx = 0
    for size in group_sizes:
        probs[idx:idx + size] = softmax(scores[idx:idx + size])
        idx += size
    return probs

def get_group_sizes(groups):
    seen = {}
    sizes = []
    for g in groups:
        if g not in seen:
            seen[g] = (groups == g).sum()
            sizes.append(seen[g])
    return np.array(sizes)

def compute_race_ndcg(y_true, scores, groups, k=3):
    ndcgs = []
    idx = 0
    seen = set()
    unique_groups = []
    for g in groups:
        if g not in seen:
            unique_groups.append(g)
            seen.add(g)
    for g in unique_groups:
        mask = groups == g
        size = mask.sum()
        if size < 2:
            idx += size
            continue
        yt = y_true[idx:idx + size]
        sc = scores[idx:idx + size]
        max_finish = yt.max()
        relevance = (max_finish + 1 - yt).reshape(1, -1)
        try:
            ndcgs.append(ndcg_score(relevance, sc.reshape(1, -1), k=k))
        except ValueError:
            pass
        idx += size
    return np.mean(ndcgs) if ndcgs else 0.0

print("Imports ready.")

In [ ]:
# ── Cell 4: Load model code from model.py ──
# This reads model.py directly so we don't duplicate 1400 lines
# Upload model.py to /content/ first (or clone the repo)

# Option A: Upload model.py
# from google.colab import files
# uploaded = files.upload()  # select model.py

# Option B: Clone repo (if public)
# !git clone https://github.com/Sixteen1-6/HorseRacing.git
# !cp HorseRacing/model.py /content/

# Verify it's there
!ls -la /content/*.py 2>/dev/null || echo "Upload model.py to /content/ first!"

## Step 1: Linear Regression Baseline
Quick baseline to compare against. Predicts finish position from numeric features only.

In [ ]:
# ── Cell 5: Linear Regression Baseline (~1 min) ──
from model import load_and_engineer_features, prepare_ranking_data, compute_race_ndcg, get_group_sizes, scores_to_probs
from sklearn.linear_model import LinearRegression
from sklearn.metrics import log_loss, brier_score_loss
import time

print("=" * 70)
print("LINEAR REGRESSION BASELINE")
print("=" * 70)

t0 = time.time()

# Load & engineer features (same pipeline as LambdaRank)
df = load_and_engineer_features(CSV_PATH)
X, y_rel, finish, race_ids, group_sizes, cat_features, dollar_odds = prepare_ranking_data(df)

# Temporal split (same as main model)
def extract_date(rid):
    parts = str(rid).split('_')
    for p in parts:
        if len(p) == 8 and p.isdigit(): return p
    return '00000000'

unique_races = np.unique(race_ids)
race_dates = np.array([extract_date(r) for r in unique_races])
sorted_races = unique_races[np.argsort(race_dates)]

n_total = len(sorted_races)
train_races = sorted_races[:int(n_total * 0.80)]
test_races = sorted_races[int(n_total * 0.80):]

train_mask = np.isin(race_ids, train_races)
test_mask = np.isin(race_ids, test_races)

X_train_lr = X[train_mask].copy()
X_test_lr = X[test_mask].copy()
finish_train = finish[train_mask]
finish_test = finish[test_mask]
rids_test = race_ids[test_mask]
odds_test = dollar_odds[test_mask] if dollar_odds is not None else None

# Use only numeric features (LinearRegression can't handle categoricals)
num_cols = X_train_lr.select_dtypes(include=np.number).columns.tolist()
X_train_num = X_train_lr[num_cols].fillna(0)
X_test_num = X_test_lr[num_cols].fillna(0)

# Train: predict finish position (lower = better)
lr = LinearRegression()
lr.fit(X_train_num, finish_train)
lr_pred = lr.predict(X_test_num)

# Invert predictions for ranking (higher score = better = lower finish)
lr_scores = -lr_pred

# Evaluate
ndcg3 = compute_race_ndcg(finish_test, lr_scores, rids_test, k=3)
ndcg5 = compute_race_ndcg(finish_test, lr_scores, rids_test, k=5)

# Win accuracy
unique_test_races = []
seen = set()
for g in rids_test:
    if g not in seen:
        unique_test_races.append(g)
        seen.add(g)

win_correct = 0
idx = 0
for rid in unique_test_races:
    mask = rids_test == rid
    size = mask.sum()
    race_scores = lr_scores[idx:idx + size]
    race_finish = finish_test[idx:idx + size]
    if np.argmax(race_scores) == np.argmin(race_finish):
        win_correct += 1
    idx += size

# ROI simulation
if odds_test is not None:
    group_sz = get_group_sizes(rids_test)
    lr_probs = scores_to_probs(lr_scores, group_sz)
    
    total_wagered = 0
    total_returned = 0
    n_bets = 0
    idx = 0
    for rid in unique_test_races:
        mask = rids_test == rid
        size = mask.sum()
        race_probs = lr_probs[idx:idx + size]
        race_probs = race_probs / race_probs.sum()
        race_finish = finish_test[idx:idx + size]
        race_odds = odds_test[idx:idx + size]
        
        for h in range(size):
            if np.isnan(race_odds[h]) or race_odds[h] <= 0: continue
            implied = 1.0 / (race_odds[h] + 1)
            if race_probs[h] > implied * 1.20:
                total_wagered += 2.0
                n_bets += 1
                if race_finish[h] == 1:
                    total_returned += 2.0 * (race_odds[h] + 1)
        idx += size
    
    lr_roi = (total_returned - total_wagered) / total_wagered * 100 if total_wagered > 0 else 0

elapsed = time.time() - t0

print(f"\n{'=' * 50}")
print(f"LINEAR REGRESSION BASELINE RESULTS")
print(f"{'=' * 50}")
print(f"  NDCG@3:        {ndcg3:.4f}")
print(f"  NDCG@5:        {ndcg5:.4f}")
print(f"  Win accuracy:  {win_correct/len(unique_test_races):.1%}")
if odds_test is not None:
    print(f"  ROI (1.2x edge): {lr_roi:+.1f}% ({n_bets} bets)")
print(f"  Time:          {elapsed:.1f}s")
print(f"{'=' * 50}")
print(f"\n  Save these numbers to compare against LambdaRank!")

# Store for comparison
baseline_results = {'ndcg3': ndcg3, 'ndcg5': ndcg5, 
                    'win_acc': win_correct/len(unique_test_races),
                    'time': elapsed}

## Step 2: LambdaRank Model (Full Training)
This runs Optuna hyperparameter search + final model training + calibration + evaluation.
- `n_trials=5` for quick test (~5 min)
- `n_trials=100` for full training (~1-2 hours)

In [ ]:
# ── Cell 6: Train LambdaRank Model ──
# Change n_trials: 5 = quick test, 100 = full training
N_TRIALS = 5  # <-- change to 100 for full training

from model import main
main(input_csv=CSV_PATH, quick=False, n_trials=N_TRIALS, output_dir=OUTPUT_DIR)

## Step 3: SHAP Analysis
Feature importance analysis showing which features drive predictions.

In [ ]:
# ── Cell 7: SHAP Analysis ──
# Run this AFTER Cell 6 completes. Loads the saved model.
import shap
import lightgbm as lgb
import joblib
import numpy as np

model_path = os.path.join(OUTPUT_DIR, 'ranking_model.lgb')
feature_names_path = os.path.join(OUTPUT_DIR, 'ranking_feature_names.pkl')

if not os.path.exists(model_path):
    print("Run Cell 6 first to train the model!")
else:
    model = lgb.Booster(model_file=model_path)
    feature_names = joblib.load(feature_names_path)
    
    # Reload data for SHAP (use test set)
    from model import load_and_engineer_features, prepare_ranking_data
    df = load_and_engineer_features(CSV_PATH)
    X, y_rel, finish, race_ids, _, cat_features, _ = prepare_ranking_data(df)
    
    # Use last 20% as test
    unique_races = np.unique(race_ids)
    test_races = unique_races[int(len(unique_races) * 0.80):]
    test_mask = np.isin(race_ids, test_races)
    X_test_shap = X[test_mask]
    
    # Sample 5000 rows for speed
    if len(X_test_shap) > 5000:
        idx = np.random.choice(len(X_test_shap), 5000, replace=False)
        X_test_shap = X_test_shap.iloc[idx]
    
    print(f"Computing SHAP values on {len(X_test_shap)} samples...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test_shap)
    
    # Summary plot
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test_shap, feature_names=feature_names, 
                      max_display=20, show=False)
    plt.title("SHAP Feature Importance (Top 20)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'shap_summary.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    # Bar plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test_shap, feature_names=feature_names,
                      plot_type="bar", max_display=20, show=False)
    plt.title("SHAP Mean Absolute Impact")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'shap_bar.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nSaved SHAP plots to {OUTPUT_DIR}/")

## Step 4: Compare Results
Side-by-side comparison of baseline vs LambdaRank.

In [ ]:
# ── Cell 8: Compare Baseline vs LambdaRank ──
import json

metrics_path = os.path.join(OUTPUT_DIR, 'ranking_metrics.json')
if not os.path.exists(metrics_path):
    print("Run Cell 6 first!")
else:
    with open(metrics_path) as f:
        lgb_metrics = json.load(f)
    
    print(f"{'=' * 60}")
    print(f"{'METRIC':<25} {'Lin.Reg Baseline':>18} {'LambdaRank':>15}")
    print(f"{'=' * 60}")
    print(f"{'NDCG@3':<25} {baseline_results['ndcg3']:>18.4f} {lgb_metrics['ndcg3']:>15.4f}")
    print(f"{'NDCG@5':<25} {baseline_results['ndcg5']:>18.4f} {lgb_metrics['ndcg5']:>15.4f}")
    print(f"{'Win Accuracy':<25} {baseline_results['win_acc']:>17.1%} {lgb_metrics['win_acc']:>14.1%}")
    if 'roi_1.2' in lgb_metrics:
        print(f"{'ROI (1.2x edge)':<25} {'N/A':>18} {lgb_metrics['roi_1.2']:>+14.1f}%")
    print(f"{'Training Time':<25} {baseline_results['time']:>17.0f}s {'~600-3600':>14}s")
    print(f"{'=' * 60}")